In [2]:
import pathlib
import scipy

In [ ]:
import pandas as pd
import scipy.stats as stats

# Load your dataset (replace with actual file path)
df = pd.read_csv("your_data.csv")

# Define the two scenarios to compare (adjust based on your dataset)
scenario_1 = "Bright"
scenario_2 = "Dark"

# Filter data for each scenario
df_s1 = df[df["scenario"] == scenario_1]
df_s2 = df[df["scenario"] == scenario_2]

# Ensure both scenarios have the same folds
common_folds = set(df_s1["FOLD"]).intersection(set(df_s2["FOLD"]))

# Compute fold-level MAE for each scenario
fold_mae_s1 = []
fold_mae_s2 = []

for fold in common_folds:
    # Get absolute errors for the current fold
    mae_s1 = df_s1[df_s1["FOLD"] == fold]["Error"].abs().mean()
    mae_s2 = df_s2[df_s2["FOLD"] == fold]["Error"].abs().mean()

    fold_mae_s1.append(mae_s1)
    fold_mae_s2.append(mae_s2)

# Convert to pandas Series for easier manipulation
fold_mae_s1 = pd.Series(fold_mae_s1)
fold_mae_s2 = pd.Series(fold_mae_s2)

# Compute the difference between paired fold MAEs
mae_diff = fold_mae_s1 - fold_mae_s2

# Step 1: Check for normality
shapiro_test = stats.shapiro(mae_diff)
p_normality = shapiro_test.pvalue

# Step 2: Choose appropriate test
if p_normality > 0.05:  # Data is normally distributed
    t_stat, p_value = stats.ttest_rel(fold_mae_s1, fold_mae_s2)
    test_used = "Paired t-test"
else:  # Non-normal distribution
    t_stat, p_value = stats.wilcoxon(fold_mae_s1, fold_mae_s2)
    test_used = "Wilcoxon signed-rank test"

# Print results
print(f"Test Used: {test_used}")
print(f"p-value: {p_value:.5f}")

if p_value < 0.05:
    print("Significant difference detected between scenarios!")
else:
    print("No significant difference detected between scenarios.")
